# Data processing pas a pas

Ce notebook rejoue `process_raw_data` sur un fichier de ton choix, mais en etapes separees pour pouvoir verifier chaque transformation.

Il montre aussi les specificites qu'on a observees dans les donnees:
- valeurs negatives sur `Quantité`, `Montant`, `PU Net`
- lignes ou `Montant` n'est pas coherent avec `Quantité * PU Net`
- impact de la normalisation finale vers l'unite canonique `Montant = Quantité * PU Net`

Si tu veux reproduire exactement l'entree actuelle de l'app sur `Combined_Sales_Data.xlsx`, mets `TRIM_LAST_ROWS = 2`.


In [1]:
from pathlib import Path

FILE_PATH = Path("Sales Data/Raw data High tech 2024.xlsx")
SHEET_NAME = 0
TRIM_LAST_ROWS = 0
DISPLAY_ROWS = 10
EXPORT_FINAL = False
OUTPUT_PATH = Path("Sales Data/processed_from_notebook.xlsx")


In [2]:
import re
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)
from openpyxl.styles.named_styles import _NamedCellStyle

_orig_init = _NamedCellStyle.__init__

def _patched_init(self, *args, **kwargs):
    if "biltinId" in kwargs:
        kwargs["builtinId"] = kwargs.pop("biltinId")
    _orig_init(self, *args, **kwargs)

_NamedCellStyle.__init__ = _patched_init

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

QTY = "Quantité"
AMOUNT = "Montant"
UNIT_PRICE = "PU Net"
CLIENT = "Cpt Client"
ARTICLE = "Code Artic"
FAMILY = "Famille"
LABEL1 = "Libelle 1"
LABEL2 = "Libelle 2"
TITLE = "Intitulé"
INVOICE_DATE = "Date Fact."
ORDER_DATE = "Date Cde"
SHIP_DATE = "Date Exp."
PLANNED_DELAY = "Delai Prev"
CURRENCY = "Nom Devise"
BON = "N° Bon"

def load_input_file(path, sheet_name=0):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls", ".xlsm"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    if suffix == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Format non supporte: {suffix}")

country_mapping = {
    **{str(i).zfill(2): "FR" for i in range(100)},
    "AE": "AE", "AT": "AT", "AU": "AU", "BE": "BE", "BG": "BG", "BR": "BR",
    "CA": "CA", "CH": "CH", "CL": "CL", "CR": "CR", "CZ": "CZ", "DE": "DE",
    "DK": "DK", "ES": "ES", "GB": "GB", "GR": "GR", "HR": "HR", "HU": "HU",
    "IE": "IE", "IN": "IN", "IT": "IT", "LT": "LT", "LU": "LU", "MK": "MK",
    "NL": "NL", "NO": "NO", "PL": "PL", "PT": "PT", "RO": "RO", "RS": "RS",
    "SA": "SA", "SE": "SE", "SK": "SK", "UY": "UY",
}

def get_country_code(client_code):
    if pd.isna(client_code):
        return "Unknown"
    prefix = str(client_code)[:2]
    if prefix.isdigit():
        return "FR"
    return country_mapping.get(prefix, "Unknown")

def extract_code(artic):
    if pd.isna(artic):
        return None
    matches = re.findall(r"([A-Z])(\d{3})", str(artic))
    if matches:
        chosen = matches[1] if len(matches) > 1 else matches[0]
        return chosen[0] + chosen[1]
    return None

step_log = []

def log_step(step_name, before, after, note=""):
    step_log.append({
        "step": step_name,
        "before": int(before),
        "after": int(after),
        "delta": int(after - before),
        "note": note,
    })
    print(f"{step_name}: {before:,} -> {after:,} lignes ({after - before:+,})")
    if note:
        print(f"  {note}")

def cols_that_exist(df, columns):
    return [col for col in columns if col in df.columns]

def financial_summary(df):
    rows = []
    for col in [QTY, AMOUNT, UNIT_PRICE]:
        if col in df.columns:
            series = pd.to_numeric(df[col], errors="coerce")
            rows.append({
                "colonne": col,
                "nan": int(series.isna().sum()),
                "negatifs": int((series < 0).sum()),
                "zeros": int((series == 0).sum()),
                "positifs": int((series > 0).sum()),
            })
    if not rows:
        return pd.DataFrame()
    return pd.DataFrame(rows).set_index("colonne")

def build_ratio_examples(df):
    required = {AMOUNT, QTY, UNIT_PRICE}
    if not required.issubset(df.columns):
        return pd.DataFrame()
    valid = df.loc[
        df[AMOUNT].notna()
        & df[QTY].notna()
        & df[UNIT_PRICE].notna()
        & (df[AMOUNT] >= 0)
        & (df[QTY] > 0)
        & (df[UNIT_PRICE] > 0)
    ].copy()
    if valid.empty:
        return valid
    valid["Montant_theorique"] = valid[QTY] * valid[UNIT_PRICE]
    valid["Ratio_montant_sur_theorique"] = valid[AMOUNT] / valid["Montant_theorique"]
    return valid

def diff_view(before_df, after_df, columns, n=10):
    if not columns:
        return pd.DataFrame()
    merged = before_df[["_source_row"] + columns].merge(
        after_df[["_source_row"] + columns],
        on="_source_row",
        how="left",
        suffixes=("_before", "_after"),
    )
    changed = pd.Series(False, index=merged.index)
    for col in columns:
        left = merged[f"{col}_before"]
        right = merged[f"{col}_after"]
        same = left.eq(right) | (left.isna() & right.isna())
        changed = changed | (~same)
    return merged.loc[changed].head(n)


In [3]:
df_raw = load_input_file(FILE_PATH, sheet_name=SHEET_NAME)
print(f"Fichier charge: {FILE_PATH}")
print(f"Lignes brutes: {len(df_raw):,}")

df = df_raw.copy()
if TRIM_LAST_ROWS > 0 and TRIM_LAST_ROWS < len(df):
    df = df.iloc[:-TRIM_LAST_ROWS].copy()
    print(f"{TRIM_LAST_ROWS} derniere(s) ligne(s) retiree(s) pour coller au pipeline de l'app.")

df["_source_row"] = np.arange(len(df))
print(f"Lignes de depart pour le notebook: {len(df):,}")
display(df.head(DISPLAY_ROWS))


Fichier charge: Sales Data\Raw data High tech 2024.xlsx
Lignes brutes: 35,646
Lignes de depart pour le notebook: 35,646


,Cpt Client,Intitulé,Code Artic,Compl Art,Libelle 1,Quantité,Montant,N° Bon,Date Exp.,Date Fact.,Nature Art,Delai Prev,Famille,Ss Famille,Date Cde,PU Net,Nom Devise,Compte Vte,Code Tarif,Code Group,Cpt Compta,Cpte vente,_source_row
0,01MIRO,MIRONLAB,FP400DP765672,E01,PREWORKOUT SPORT AROME FRUITS,0.00,0.00,071228,2024-03-11,NaT,6,2024-01-22,PDRE,672,2023-12-12,5.73,EURO,701RRX,0,NaN,NaN,NaN,0
1,01MIRO,MIRONLAB,FP400DP765672,E01,PREWORKOUT SPORT AROME FRUITS,708.00,"4,056.13",071228,2024-03-11,2024-03-11,6,2024-01-22,PDRE,672,2023-12-12,5.73,EURO,701RRX,0,NaN,NaN,NaN,1
2,01MIRO,MIRONLAB,FP400DP765672,E01,PREWORKOUT SPORT AROME FRUITS,"1,728.00","9,899.71",072325,2024-03-11,2024-03-11,6,2024-03-08,PDRE,672,2024-03-08,5.73,EURO,701RRX,0,NaN,NaN,NaN,2
3,03COMA,SQUAD NUTRITION SAS,FK15BT626878,E010,FLUFFY VANILLA FLAVOURED CRISP,"10,638.00","9,042.30",073004,2024-11-29,2024-11-29,6,2024-11-29,BARS,878,2024-04-30,850.00,EURO,701RRX,0,NaN,NaN,NaN,3
4,03COMA,SQUAD NUTRITION SAS,FK15BT627878,E01,FLUFFY CHOCOLATE FLAVOURED,"9,972.00","8,476.20",073004,2024-12-17,2024-12-17,6,2024-12-20,BARS,878,2024-04-30,850.00,EURO,701RRX,0,NaN,NaN,NaN,4
5,03COMA,NOE MARQUEZ,FS10NC335302,E01,MILK CHOCOLATE COATED,140.00,0.00,073087,2024-05-15,NaT,6,2024-05-06,BUZI,302,2024-05-06,0.00,EURO,701RRX,A,NaN,NaN,NaN,5
6,03COMA,NOE MARQUEZ,FS10NC336302,E01,WHITE CHOCOLATE COATED,140.00,0.00,073087,2024-05-15,NaT,6,2024-05-06,BUZI,302,2024-05-06,0.00,EURO,701RRX,A,NaN,NaN,NaN,6
7,03COMA,NOE MARQUEZ,FS15MO380302C100,E02,SACHET MUESLI CHOCOLAT CARAMEL,100.00,0.00,073087,2024-05-15,NaT,6,2024-05-06,MUES,302,2024-05-06,0.00,EURO,701RRX,A,NaN,NaN,NaN,7
8,03COMA,NOE MARQUEZ,FK15BT626302,E01,FLUFFY VANILLA FLAVOURED CRISP,120.00,0.00,073087,2024-05-15,NaT,6,2024-05-06,BARS,302,2024-05-06,0.00,EURO,701RRX,A,NaN,NaN,NaN,8
9,03COMA,NOE MARQUEZ,FK15BT627302,E01,FLUFFY CHOCOLATE FLAVOURED,120.00,0.00,073087,2024-05-15,NaT,6,2024-05-06,BARS,302,2024-05-06,0.00,EURO,701RRX,A,NaN,NaN,NaN,9


In [4]:
# Calcul nombre d'éléments pour une famille

print("Calcul du nombre d'elements par famille...")

for value in df[FAMILY].value_counts().index:
    print(f"Famille {value}: {df[FAMILY].eq(value).sum():,} lignes, soit {df[FAMILY].eq(value).mean():.2%} du total")

Calcul du nombre d'elements par famille...
Famille PDRE: 13,324 lignes, soit 37.38% du total
Famille BARS: 8,017 lignes, soit 22.49% du total
Famille GOFR: 4,091 lignes, soit 11.48% du total
Famille UHT: 3,371 lignes, soit 9.46% du total
Famille BUZI: 1,930 lignes, soit 5.41% du total
Famille BKRY: 1,569 lignes, soit 4.40% du total
Famille CHIP: 1,484 lignes, soit 4.16% du total
Famille MUES: 1,191 lignes, soit 3.34% du total
Famille PARI: 262 lignes, soit 0.74% du total
Famille STIC: 209 lignes, soit 0.59% du total
Famille PAIN: 90 lignes, soit 0.25% du total
Famille FAC: 59 lignes, soit 0.17% du total
Famille DEVL: 31 lignes, soit 0.09% du total
Famille POUD: 13 lignes, soit 0.04% du total
Famille ZDIV: 2 lignes, soit 0.01% du total
Famille LIQ: 1 lignes, soit 0.00% du total
Famille ========================================: 1 lignes, soit 0.00% du total


In [5]:
# Step 1 - Country
before = len(df)
if CLIENT in df.columns:
    df["Country"] = df[CLIENT].apply(get_country_code)
else:
    df["Country"] = "Unknown"

log_step("Step 1 - Country", before, len(df))
display(df[cols_that_exist(df, [CLIENT, "Country"])].head(DISPLAY_ROWS))
display(df["Country"].value_counts(dropna=False).head(15).to_frame("nb_lignes"))


Step 1 - Country: 35,646 -> 35,646 lignes (+0)


,Cpt Client,Country
0,01MIRO,FR
1,01MIRO,FR
2,01MIRO,FR
3,03COMA,FR
4,03COMA,FR
5,03COMA,FR
6,03COMA,FR
7,03COMA,FR
8,03COMA,FR
9,03COMA,FR


,nb_lignes
Country,
FR,10786
NL,5774
BE,5414
IT,2662
IE,2214
ES,1848
GB,1247
SK,920
CZ,902


In [6]:
# Step 2 - Conversions
before = len(df)
for col in [SHIP_DATE, INVOICE_DATE, ORDER_DATE, PLANNED_DELAY]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
for col in [QTY, AMOUNT, UNIT_PRICE]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

log_step("Step 2 - Conversions", before, len(df))
display(financial_summary(df))
display(df[cols_that_exist(df, [INVOICE_DATE, ORDER_DATE, SHIP_DATE, PLANNED_DELAY, QTY, AMOUNT, UNIT_PRICE])].head(DISPLAY_ROWS))


Step 2 - Conversions: 35,646 -> 35,646 lignes (+0)


,nan,negatifs,zeros,positifs
colonne,,,,
Quantité,0,135,6584,28927
Montant,0,284,9282,26080
PU Net,0,0,3230,32416


,Date Fact.,Date Cde,Date Exp.,Delai Prev,Quantité,Montant,PU Net
0,NaT,2023-12-12,2024-03-11,2024-01-22,0.00,0.00,5.73
1,2024-03-11,2023-12-12,2024-03-11,2024-01-22,708.00,"4,056.13",5.73
2,2024-03-11,2024-03-08,2024-03-11,2024-03-08,"1,728.00","9,899.71",5.73
3,2024-11-29,2024-04-30,2024-11-29,2024-11-29,"10,638.00","9,042.30",850.00
4,2024-12-17,2024-04-30,2024-12-17,2024-12-20,"9,972.00","8,476.20",850.00
5,NaT,2024-05-06,2024-05-15,2024-05-06,140.00,0.00,0.00
6,NaT,2024-05-06,2024-05-15,2024-05-06,140.00,0.00,0.00
7,NaT,2024-05-06,2024-05-15,2024-05-06,100.00,0.00,0.00
8,NaT,2024-05-06,2024-05-15,2024-05-06,120.00,0.00,0.00
9,NaT,2024-05-06,2024-05-15,2024-05-06,120.00,0.00,0.00


In [7]:
# Steps 3, 4, 5
before = len(df)
cols_to_drop = ["Code Group", "Cpt Compta", "Cpte vente"]
existing_cols_to_drop = [col for col in cols_to_drop if col in df.columns]
if existing_cols_to_drop:
    df.drop(existing_cols_to_drop, axis=1, inplace=True)

if ARTICLE in df.columns:
    df["Code Recette"] = df[ARTICLE].apply(extract_code)
else:
    df["Code Recette"] = None

if ORDER_DATE in df.columns:
    df.sort_values(ORDER_DATE, inplace=True)

if LABEL1 in df.columns and ARTICLE in df.columns:
    df[LABEL1] = df.groupby(ARTICLE)[LABEL1].transform("first")
elif LABEL1 not in df.columns:
    df[LABEL1] = None

log_step(
    "Step 3 - Colonnes et enrichissement",
    before,
    len(df),
    note=(f"Colonnes supprimees: {', '.join(existing_cols_to_drop)}" if existing_cols_to_drop else "Aucune colonne a supprimer"),
)

duplicates = int(df.duplicated().sum())
log_step(
    "Step 4 - Doublons",
    len(df),
    len(df),
    note=f"{duplicates:,} doublons exacts detectes, aucune suppression pour rester aligne avec process_raw_data.",
)

before = len(df)
familles_exclues = ["LIQ", "DEVL", "ZDIV", "EMB"]
if FAMILY in df.columns:
    removed_step_5 = df.loc[df[FAMILY].isin(familles_exclues)].copy()
    df = df.loc[~df[FAMILY].isin(familles_exclues)].copy()
else:
    removed_step_5 = df.iloc[0:0].copy()

log_step("Step 5 - Familles exclues", before, len(df), note="Familles exclues: LIQ, DEVL, ZDIV, EMB")
if not removed_step_5.empty:
    display(removed_step_5[FAMILY].value_counts().rename_axis(FAMILY).to_frame("nb_lignes"))
display(df[cols_that_exist(df, [ARTICLE, "Code Recette", LABEL1, ORDER_DATE, FAMILY])].head(DISPLAY_ROWS))


Step 3 - Colonnes et enrichissement: 35,646 -> 35,646 lignes (+0)
  Colonnes supprimees: Code Group, Cpt Compta, Cpte vente
Step 4 - Doublons: 35,646 -> 35,646 lignes (+0)
  0 doublons exacts detectes, aucune suppression pour rester aligne avec process_raw_data.
Step 5 - Familles exclues: 35,646 -> 35,612 lignes (-34)
  Familles exclues: LIQ, DEVL, ZDIV, EMB


,nb_lignes
Famille,
DEVL,31
ZDIV,2
LIQ,1


,Code Artic,Code Recette,Libelle 1,Date Cde,Famille
21455,FK15BT617302,T617,DOUBLE CHOCOLATE CARAMEL,2023-03-06,BARS
21456,FS15NC331302,C331,BULLES CHOCOLAT (35 G),2023-03-20,BUZI
8795,FK15BC306806,C306,BARRES DOUBLE CHOCOLAT (42G),2023-03-20,BARS
8794,FK15BC306806,C306,BARRES DOUBLE CHOCOLAT (42G),2023-03-20,BARS
8793,FK15BT617806,T617,DOUBLE CHOCOLATE CARAMEL,2023-03-20,BARS
8792,FK15BC306806,C306,BARRES DOUBLE CHOCOLAT (42G),2023-03-20,BARS
8791,FK15BC306806,C306,BARRES DOUBLE CHOCOLAT (42G),2023-03-20,BARS
16475,FK11.3BT800659,T800,BARRE BIO FIGUES ET AMANDES,2023-05-04,BARS
16476,FK11.3BT800659,T800,BARRE BIO FIGUES ET AMANDES,2023-05-04,BARS
16477,FK11.3BT800659,T800,BARRE BIO FIGUES ET AMANDES,2023-05-04,BARS


In [8]:
# Step 6 + diagnostics avant corrections
before = len(df)
dropped_nat_fact = 0

if INVOICE_DATE in df.columns:
    dropped_nat_fact = int(df[INVOICE_DATE].isna().sum())
    if dropped_nat_fact > 0:
        df = df.dropna(subset=[INVOICE_DATE]).copy()
    df.sort_values(INVOICE_DATE, inplace=True)
    df.set_index(INVOICE_DATE, drop=False, inplace=True)
    df["Date_Ref"] = df.index
    df["Year"] = df.index.year
    df["Month"] = df.index.to_period("M").astype(str)
    df["Quarter"] = df.index.to_period("Q-MAR").astype(str)
    df["Fiscal_Year"] = df.index.year + (df.index.month >= 4).astype(int)
    df["Fiscal_Year_Label"] = "FY" + df["Fiscal_Year"].astype(str)
    quarter_labels = {1: "Q1 (Avr-Juin)", 2: "Q2 (Juil-Sep)", 3: "Q3 (Oct-Dec)", 4: "Q4 (Jan-Mar)"}
    df["Trimestre_Num"] = ((df.index.month - 4) % 12) // 3 + 1
    df["Fiscal_Quarter"] = df["Trimestre_Num"].map(quarter_labels) + " " + df["Fiscal_Year_Label"]
    df["Weekday"] = df.index.weekday
else:
    df["Year"] = np.nan
    df["Month"] = None
    df["Quarter"] = None
    df["Fiscal_Year"] = np.nan
    df["Fiscal_Year_Label"] = None
    df["Trimestre_Num"] = np.nan
    df["Fiscal_Quarter"] = None
    df["Weekday"] = np.nan

if SHIP_DATE in df.columns and ORDER_DATE in df.columns:
    df["Lead_Time_Days"] = (df[SHIP_DATE] - df[ORDER_DATE]).dt.days
else:
    df["Lead_Time_Days"] = np.nan

if PLANNED_DELAY in df.columns and ORDER_DATE in df.columns:
    df["Delai_Prev_Days"] = (df[PLANNED_DELAY] - df[ORDER_DATE]).dt.days
else:
    df["Delai_Prev_Days"] = np.nan

df["Lead_Time_Deviation_Days"] = df["Lead_Time_Days"] - df["Delai_Prev_Days"]

log_step("Step 6 - Variables temporelles", before, len(df), note=f"Lignes retirees car Date Fact. vide: {dropped_nat_fact:,}")

if {AMOUNT, QTY, UNIT_PRICE}.issubset(df.columns):
    df["_pu_reference"] = np.nan
    positive_pu_mask = df[UNIT_PRICE].notna() & (df[UNIT_PRICE] > 0)
    if ARTICLE in df.columns:
        code_pu_reference = df.loc[positive_pu_mask].groupby(ARTICLE)[UNIT_PRICE].median()
        df["_pu_reference"] = df[ARTICLE].map(code_pu_reference)
    if FAMILY in df.columns:
        family_pu_reference = df.loc[positive_pu_mask].groupby(FAMILY)[UNIT_PRICE].median()
        df["_pu_reference"] = df["_pu_reference"].fillna(df[FAMILY].map(family_pu_reference))

print("Resume financier avant steps 7/8")
display(financial_summary(df))

ratio_examples = build_ratio_examples(df)
if not ratio_examples.empty:
    signature = pd.Series("autre", index=ratio_examples.index)
    signature = signature.mask(np.isclose(ratio_examples["Ratio_montant_sur_theorique"], 1.0, atol=0.01), "~1")
    signature = signature.mask(np.isclose(ratio_examples["Ratio_montant_sur_theorique"], 0.001, atol=0.0002), "~0.001")
    print("Signatures du ratio Montant / (Quantité * PU Net)")
    display(signature.value_counts().rename_axis("signature").to_frame("nb_lignes"))

    suspicious = ratio_examples.loc[
        ratio_examples["Ratio_montant_sur_theorique"] < 0.01,
        cols_that_exist(ratio_examples, ["_source_row", ARTICLE, FAMILY, QTY, UNIT_PRICE, AMOUNT, "Montant_theorique", "Ratio_montant_sur_theorique"]),
    ]
    if not suspicious.empty:
        print("Exemples de lignes ou le Montant source est tres en dessous de Quantité * PU Net")
        display(suspicious.head(DISPLAY_ROWS))

    direct_examples = ratio_examples.loc[
        np.isclose(ratio_examples["Ratio_montant_sur_theorique"], 1.0, atol=0.01),
        cols_that_exist(ratio_examples, ["_source_row", ARTICLE, FAMILY, QTY, UNIT_PRICE, AMOUNT, "Montant_theorique", "Ratio_montant_sur_theorique"]),
    ]
    if not direct_examples.empty:
        print("Exemples de lignes deja coherentes avec Quantité * PU Net")
        display(direct_examples.head(DISPLAY_ROWS))

display(pd.DataFrame({
    "Quantité_negatives": [int((df[QTY] < 0).sum()) if QTY in df.columns else 0],
    "Montant_negatifs": [int((df[AMOUNT] < 0).sum()) if AMOUNT in df.columns else 0],
    "PU_Net_negatifs": [int((df[UNIT_PRICE] < 0).sum()) if UNIT_PRICE in df.columns else 0],
    "Quantité_zero": [int((df[QTY] == 0).sum()) if QTY in df.columns else 0],
    "Montant_zero": [int((df[AMOUNT] == 0).sum()) if AMOUNT in df.columns else 0],
}))


Step 6 - Variables temporelles: 35,612 -> 29,718 lignes (-5,894)
  Lignes retirees car Date Fact. vide: 5,894
Resume financier avant steps 7/8


,nan,negatifs,zeros,positifs
colonne,,,,
Quantité,0,135,3443,26140
Montant,0,284,3384,26050
PU Net,0,0,161,29557


Signatures du ratio Montant / (Quantité * PU Net)


,nb_lignes
signature,
~0.001,15768
~1,8743
autre,1485


Exemples de lignes ou le Montant source est tres en dessous de Quantité * PU Net


,_source_row,Code Artic,Famille,Quantité,PU Net,Montant,Montant_theorique,Ratio_montant_sur_theorique
Date Fact.,,,,,,,,
2023-12-21,14948,FK15BT627302,BARS,168.00,850.00,142.80,"142,800.00",0.00
2024-01-02,24817,FW15WN604302,GOFR,64.00,"1,078.00",68.99,"68,992.00",0.00
2024-01-02,24802,FT15SETU008302,UHT,16.00,"1,150.00",18.40,"18,400.00",0.00
2024-01-02,5746,FK15BT620302,BARS,200.00,990.00,198.00,"198,000.00",0.00
2024-01-02,5745,FK15BT621302,BARS,200.00,970.00,194.00,"194,000.00",0.00
2024-01-02,24819,FW15WN608302,GOFR,64.00,"1,078.00",68.99,"68,992.00",0.00
2024-01-02,24816,FW15WN603302,GOFR,64.00,"1,075.00",68.80,"68,800.00",0.00
2024-01-02,24809,FS5RG910302,BKRY,400.00,450.00,180.00,"180,000.00",0.00
2024-01-02,24810,FW15WN607302,GOFR,64.00,"1,042.00",66.69,"66,688.00",0.00


Exemples de lignes deja coherentes avec Quantité * PU Net


,_source_row,Code Artic,Famille,Quantité,PU Net,Montant,Montant_theorique,Ratio_montant_sur_theorique
Date Fact.,,,,,,,,
2023-12-21,14944,FP100DP746302,PDRE,28.00,2.40,67.20,67.20,1.00
2023-12-21,14930,FS18DP791302C150,PDRE,112.00,0.53,59.36,59.36,1.00
2023-12-21,14928,FS18FF259302C150,PDRE,112.00,0.54,60.48,60.48,1.00
2023-12-21,14954,FS18EF810302C150,PDRE,112.00,0.53,59.36,59.36,1.00
2023-12-21,14945,FP100DP748302,PDRE,28.00,2.40,67.20,67.20,1.00
2023-12-21,14947,FP100DP755302,PDRE,28.00,2.40,67.20,67.20,1.00
2023-12-21,14929,FS18OF183302C150,PDRE,112.00,0.47,52.64,52.64,1.00
2023-12-21,14938,FS18SF918302C150,PDRE,112.00,0.48,53.76,53.76,1.00
2023-12-21,14950,FS18MO361302C150,PDRE,112.00,0.50,56.00,56.00,1.00


,Quantité_negatives,Montant_negatifs,PU_Net_negatifs,Quantité_zero,Montant_zero
0,135,284,0,3443,3384


In [9]:
# Step 7 - Reparation des negatifs puis suppression des negatifs restants
before = len(df)
df_before_step_7 = df.copy()
qty_fixed = amount_fixed = pu_fixed = 0
removed_qty_rows = df.iloc[0:0].copy()
removed_amount_rows = df.iloc[0:0].copy()

if {QTY, AMOUNT, UNIT_PRICE}.issubset(df.columns):
    qty_invalid_but_recoverable = (
        df[QTY].notna() & (df[QTY] < 0)
        & df[AMOUNT].notna() & (df[AMOUNT] >= 0)
        & df[UNIT_PRICE].notna() & (df[UNIT_PRICE] > 0)
    )
    qty_fixed = int(qty_invalid_but_recoverable.sum())
    df.loc[qty_invalid_but_recoverable, QTY] = df.loc[qty_invalid_but_recoverable, AMOUNT] / df.loc[qty_invalid_but_recoverable, UNIT_PRICE]

    amount_invalid_but_recoverable = (
        df[AMOUNT].notna() & (df[AMOUNT] < 0)
        & df[QTY].notna() & (df[QTY] >= 0)
        & df[UNIT_PRICE].notna() & (df[UNIT_PRICE] >= 0)
    )
    amount_fixed = int(amount_invalid_but_recoverable.sum())
    df.loc[amount_invalid_but_recoverable, AMOUNT] = df.loc[amount_invalid_but_recoverable, QTY] * df.loc[amount_invalid_but_recoverable, UNIT_PRICE]

    pu_invalid_but_recoverable = (
        df[UNIT_PRICE].notna() & (df[UNIT_PRICE] < 0)
        & df[AMOUNT].notna() & (df[AMOUNT] >= 0)
        & df[QTY].notna() & (df[QTY] != 0) & (df[QTY] >= 0)
    )
    pu_fixed = int(pu_invalid_but_recoverable.sum())
    df.loc[pu_invalid_but_recoverable, UNIT_PRICE] = df.loc[pu_invalid_but_recoverable, AMOUNT] / df.loc[pu_invalid_but_recoverable, QTY]

if QTY in df.columns:
    invalid_qty_mask = df[QTY].notna() & (df[QTY] < 0)
    removed_qty_rows = df.loc[invalid_qty_mask].copy()
    df = df.loc[~invalid_qty_mask].copy()

if AMOUNT in df.columns:
    invalid_amount_mask = df[AMOUNT].notna() & (df[AMOUNT] < 0)
    removed_amount_rows = df.loc[invalid_amount_mask].copy()
    df = df.loc[~invalid_amount_mask].copy()

log_step(
    "Step 7 - Negatifs",
    before,
    len(df),
    note=f"Quantité reparee: {qty_fixed:,} | Montant repare: {amount_fixed:,} | PU Net repare: {pu_fixed:,}",
)
print(f"Lignes supprimees car Quantité reste negative: {len(removed_qty_rows):,}")
print(f"Lignes supprimees car Montant reste negatif: {len(removed_amount_rows):,}")

step_7_changes = diff_view(df_before_step_7, df, cols_that_exist(df_before_step_7, [ARTICLE, FAMILY, QTY, AMOUNT, UNIT_PRICE]), n=DISPLAY_ROWS)
if not step_7_changes.empty:
    print("Exemples de lignes touchees par le step 7")
    display(step_7_changes)


Step 7 - Negatifs: 29,718 -> 29,583 lignes (-135)
  Quantité reparee: 0 | Montant repare: 149 | PU Net repare: 0
Lignes supprimees car Quantité reste negative: 135
Lignes supprimees car Montant reste negatif: 0
Exemples de lignes touchees par le step 7


,_source_row,Code Artic_before,Famille_before,Quantité_before,Montant_before,PU Net_before,Code Artic_after,Famille_after,Quantité_after,Montant_after,PU Net_after
518,4705,FP500OF902266,PDRE,0.00,-192.00,1.60,FP500OF902266,PDRE,0.00,0.00,1.60
1258,21088,FK15BD370302,BARS,0.00,"-2,018.38",910.00,FK15BD370302,BARS,0.00,0.00,910.00
1301,21087,FK15BD335302,BARS,0.00,"-1,555.42",830.00,FK15BD335302,BARS,0.00,0.00,830.00
1336,13449,FT15SETU009302,UHT,0.00,"-3,427.20","1,050.00",FT15SETU009302,UHT,0.00,0.00,"1,050.00"
1419,16043,FB20SEDP791302,PDRE,0.00,-4.35,0.87,FB20SEDP791302,PDRE,0.00,0.00,0.87
1420,16474,FK15BT622659,BARS,0.00,-39.50,790.00,FK15BT622659,BARS,0.00,0.00,790.00
1448,3738,FS15NC331302,BUZI,-20.00,-16.02,801.00,NaN,NaN,NaN,NaN,NaN
1449,26130,FW15WT436302,BARS,0.00,"-3,979.92","1,030.00",FW15WT436302,BARS,0.00,0.00,"1,030.00"
1453,3739,FS16.7MO401302C100,MUES,-14.00,-7.63,0.55,NaN,NaN,NaN,NaN,NaN
1491,26129,FT15SETU009302,UHT,0.00,-595.35,"1,050.00",FT15SETU009302,UHT,0.00,0.00,"1,050.00"


In [10]:
# Step 8 - Normalisation vers Montant = Quantité * PU Net
before = len(df)
df_before_step_8 = df.copy()
pu_recalculated = amount_recalculated = qty_recalculated = 0

if {AMOUNT, QTY, UNIT_PRICE}.issubset(df.columns):
    pu_from_amount_qty = df[AMOUNT] / df[QTY]
    current_pu_gap = (df[UNIT_PRICE] - df["_pu_reference"]).abs()
    calculated_pu_gap = (pu_from_amount_qty - df["_pu_reference"]).abs()
    pu_incoherent_mask = (
        df[AMOUNT].notna() & (df[AMOUNT] >= 0)
        & df[QTY].notna() & (df[QTY] > 0)
        & (
            df[UNIT_PRICE].isna()
            | (df[UNIT_PRICE] < 0)
            | (
                df["_pu_reference"].notna()
                & calculated_pu_gap.notna()
                & (current_pu_gap.isna() | (calculated_pu_gap + 0.05 < current_pu_gap * 0.1))
            )
        )
    )
    pu_recalculated = int(pu_incoherent_mask.sum())
    df.loc[pu_incoherent_mask, UNIT_PRICE] = pu_from_amount_qty[pu_incoherent_mask]

    amount_from_qty_pu = df[QTY] * df[UNIT_PRICE]
    amount_gap = (df[AMOUNT] - amount_from_qty_pu).abs()
    amount_incoherent_mask = (
        df[QTY].notna() & (df[QTY] > 0)
        & df[UNIT_PRICE].notna() & (df[UNIT_PRICE] > 0)
        & (
            df[AMOUNT].isna()
            | (df[AMOUNT] < 0)
            | ~np.isclose(df[AMOUNT], amount_from_qty_pu, rtol=1e-6, atol=0.05)
            | (amount_gap / amount_from_qty_pu.abs().clip(lower=1.0) > 0.01)
        )
    )
    amount_recalculated = int(amount_incoherent_mask.sum())
    df.loc[amount_incoherent_mask, AMOUNT] = amount_from_qty_pu[amount_incoherent_mask]

    qty_from_amount_pu = df[AMOUNT] / df[UNIT_PRICE]
    qty_incoherent_mask = (
        df[AMOUNT].notna() & (df[AMOUNT] >= 0)
        & df[UNIT_PRICE].notna() & (df[UNIT_PRICE] > 0)
        & (df[QTY].isna() | (df[QTY] < 0))
    )
    qty_recalculated = int(qty_incoherent_mask.sum())
    df.loc[qty_incoherent_mask, QTY] = qty_from_amount_pu[qty_incoherent_mask]

log_step(
    "Step 8 - Recalcul / normalisation",
    before,
    len(df),
    note=f"PU Net recalcules: {pu_recalculated:,} | Montants recalcules: {amount_recalculated:,} | Quantités recalculees: {qty_recalculated:,}",
)

step_8_changes = diff_view(df_before_step_8, df, cols_that_exist(df_before_step_8, [ARTICLE, FAMILY, QTY, AMOUNT, UNIT_PRICE]), n=DISPLAY_ROWS)
if not step_8_changes.empty:
    print("Exemples de lignes modifiees par le step 8")
    display(step_8_changes)

if {AMOUNT, QTY, UNIT_PRICE}.issubset(df.columns):
    totals = pd.DataFrame({
        "Montant_avant_step_8": [df_before_step_8[AMOUNT].sum(skipna=True)],
        "Montant_apres_step_8": [df[AMOUNT].sum(skipna=True)],
        "Somme_theorique_Quantité_x_PU": [(df[QTY] * df[UNIT_PRICE]).sum(skipna=True)],
    })
    print("Impact global de la normalisation d'unite")
    display(totals)

    if ARTICLE in df.columns:
        code_focus = df.loc[
            df[ARTICLE].astype(str).isin(["ETUI302FZ", "HOUSSETHERMIQUE"]),
            cols_that_exist(df, ["_source_row", ARTICLE, FAMILY, QTY, UNIT_PRICE, AMOUNT]),
        ]
        if not code_focus.empty:
            print("Exemples de codes qu'on avait reperes dans les donnees")
            display(code_focus.head(DISPLAY_ROWS))


Step 8 - Recalcul / normalisation: 29,583 -> 29,583 lignes (+0)
  PU Net recalcules: 2,723 | Montants recalcules: 16,385 | Quantités recalculees: 0
Exemples de lignes modifiees par le step 8


,_source_row,Code Artic_before,Famille_before,Quantité_before,Montant_before,PU Net_before,Code Artic_after,Famille_after,Quantité_after,Montant_after,PU Net_after
8,14948,FK15BT627302,BARS,168.00,142.80,850.00,FK15BT627302,BARS,168.00,"142,800.00",850.00
29,24817,FW15WN604302,GOFR,64.00,68.99,"1,078.00",FW15WN604302,GOFR,64.00,"68,992.00","1,078.00"
30,24802,FT15SETU008302,UHT,16.00,18.40,"1,150.00",FT15SETU008302,UHT,16.00,"18,400.00","1,150.00"
33,5746,FK15BT620302,BARS,200.00,198.00,990.00,FK15BT620302,BARS,200.00,"198,000.00",990.00
35,5745,FK15BT621302,BARS,200.00,194.00,970.00,FK15BT621302,BARS,200.00,"194,000.00",970.00
39,24819,FW15WN608302,GOFR,64.00,68.99,"1,078.00",FW15WN608302,GOFR,64.00,"68,992.00","1,078.00"
42,24816,FW15WN603302,GOFR,64.00,68.80,"1,075.00",FW15WN603302,GOFR,64.00,"68,800.00","1,075.00"
43,24809,FS5RG910302,BKRY,400.00,180.00,450.00,FS5RG910302,BKRY,400.00,"180,000.00",450.00
44,24810,FW15WN607302,GOFR,64.00,66.69,"1,042.00",FW15WN607302,GOFR,64.00,"66,688.00","1,042.00"
46,20863,FS18OF183302C150,PDRE,224.00,126.12,0.49,FS18OF183302C150,PDRE,224.00,109.76,0.49


Impact global de la normalisation d'unite


,Montant_avant_step_8,Montant_apres_step_8,Somme_theorique_Quantité_x_PU
0,"39,783,974.60","20,351,192,914.60","20,351,112,818.64"


In [11]:
# Step 9 / 10 - Dataset final
colonnes_finales = [
    INVOICE_DATE, ORDER_DATE, SHIP_DATE, "Date_Ref", "Year", "Fiscal_Year", "Fiscal_Year_Label",
    "Month", "Quarter", "Weekday", "Trimestre_Num", "Fiscal_Quarter", CLIENT, TITLE,
    "Country", "Code Recette", LABEL1, LABEL2, FAMILY, AMOUNT, UNIT_PRICE, QTY, CURRENCY,
    BON, "Lead_Time_Days", "Delai_Prev_Days", "Lead_Time_Deviation_Days",
]

colonnes_existantes = [col for col in colonnes_finales if col in df.columns]
df_final = df[colonnes_existantes].copy()

print(f"Lignes finales: {len(df_final):,}")
display(pd.DataFrame(step_log))
display(df_final.head(DISPLAY_ROWS))

if {AMOUNT, QTY, UNIT_PRICE}.issubset(df_final.columns):
    final_kpis = pd.DataFrame({
        "Montant_total_final": [df_final[AMOUNT].sum(skipna=True)],
        "Quantité_totale_finale": [df_final[QTY].sum(skipna=True)],
        "PU_Net_median_final": [df_final[UNIT_PRICE].median(skipna=True)],
    })
    display(final_kpis)

if EXPORT_FINAL:
    suffix = OUTPUT_PATH.suffix.lower()
    if suffix in {".xlsx", ".xls", ".xlsm"}:
        df_final.to_excel(OUTPUT_PATH, index=False)
    elif suffix == ".csv":
        df_final.to_csv(OUTPUT_PATH, index=False)
    else:
        raise ValueError(f"Format de sortie non supporte: {OUTPUT_PATH.suffix}")
    print(f"Export ecrit vers: {OUTPUT_PATH}")


Lignes finales: 29,583


,step,before,after,delta,note
0,Step 1 - Country,35646,35646,0,
1,Step 2 - Conversions,35646,35646,0,
2,Step 3 - Colonnes et enrichissement,35646,35646,0,"Colonnes supprimees: Code Group, Cpt Compta, C..."
3,Step 4 - Doublons,35646,35646,0,"0 doublons exacts detectes, aucune suppression..."
4,Step 5 - Familles exclues,35646,35612,-34,"Familles exclues: LIQ, DEVL, ZDIV, EMB"
5,Step 6 - Variables temporelles,35612,29718,-5894,"Lignes retirees car Date Fact. vide: 5,894"
6,Step 7 - Negatifs,29718,29583,-135,Quantité reparee: 0 | Montant repare: 149 | PU...
7,Step 8 - Recalcul / normalisation,29583,29583,0,"PU Net recalcules: 2,723 | Montants recalcules..."


,Date Fact.,Date Cde,Date Exp.,Date_Ref,Year,Fiscal_Year,Fiscal_Year_Label,Month,Quarter,Weekday,Trimestre_Num,Fiscal_Quarter,Cpt Client,Intitulé,Country,Code Recette,Libelle 1,Famille,Montant,PU Net,Quantité,Nom Devise,N° Bon,Lead_Time_Days,Delai_Prev_Days,Lead_Time_Deviation_Days
Date Fact.,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,L949,SACHET SOUPE CURRY NOODLES MRP,PDRE,0.00,0.47,0.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,P746,POT LEMON ORANGE WATER,PDRE,67.20,2.40,28.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,C150,SACHET BOISSON ORANGE(22.5G),PDRE,59.36,0.53,112.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,C150,SACHET FLAN CHOCOLAT (25.5G),PDRE,60.48,0.54,112.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,C150,SACHET ENTREMET CARAMEL (25G),PDRE,59.36,0.53,112.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,P748,POT ORANGE WATER,PDRE,67.20,2.40,28.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,P747,BLACK CURRANT WATER (100G),PDRE,0.00,2.40,0.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,P755,TROPICAL PUNCH (100G),PDRE,67.20,2.40,28.00,EURO,071262,21.00,8.00,13.00
2023-12-21,2023-12-21,2023-12-14,2024-01-04,2023-12-21,2023,2024,FY2024,2023-12,2024Q3,3,3,Q3 (Oct-Dec) FY2024,BEVDBN,VDB-Nutrition nv,BE,T627,FLUFFY CHOCOLATE FLAVOURED,BARS,"142,800.00",850.00,168.00,EURO,071262,21.00,8.00,13.00


,Montant_total_final,Quantité_totale_finale,PU_Net_median_final
0,"20,351,192,914.60","41,865,186.71",518.00
